# ATE Phase 1 — Tạo nhãn giả trên dataset lớn

**Pipeline:** Gate Classifier → ATE Model → Lọc high/low confidence

**Thresholds:**
- gate_threshold = 0.90
- span_threshold = 0.85

**Điều kiện high confidence:**
- gate_confidence <= 0.10 → negative signal (aspects = [])
- gate_confidence >= 0.90 AND len(aspects) >= 1 → positive signal

In [1]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [2]:
import os

GATE_MODEL_PATH = "/content/drive/MyDrive/absa_self_train_phase1/ate_gate_teacher_phase1"
ATE_MODEL_PATH  = "/content/drive/MyDrive/absa_self_train_phase1/ate_teacher_phase1"

gate_ok = os.path.isdir(GATE_MODEL_PATH)
ate_ok  = os.path.isdir(ATE_MODEL_PATH)

print("Gate model:", " OK" if gate_ok else " NOT FOUND —", GATE_MODEL_PATH)
print("ATE  model:", " OK" if ate_ok  else " NOT FOUND —", ATE_MODEL_PATH)

assert gate_ok and ate_ok, "Model không tìm thấy"

Gate model:  OK /content/drive/MyDrive/absa_self_train_phase1/ate_gate_teacher_phase1
ATE  model:  OK /content/drive/MyDrive/absa_self_train_phase1/ate_teacher_phase1


In [3]:

#Split 0--> 9
SPLIT_ID  = 4
N_SPLITS  = 15


SCRIPT_DIR   = "/content/drive/MyDrive/kindle_label"
SPLITS_DIR   = "/content/drive/MyDrive/kindle_label/split_assignment"
OUTPUT_DIR   = f"/content/drive/MyDrive/kindle_label/ATE_Phase_1/split_{SPLIT_ID:02d}"
REPORT_DIR   = f"{OUTPUT_DIR}/reports/ate_1"

# Tham số inference
GATE_THRESHOLD  = 0.90
SPAN_THRESHOLD  = 0.85
GATE_BATCH_SIZE = 128
MAX_LENGTH      = 192

NEGATIVE_UPPER = 0.10
POSITIVE_LOWER = 0.90

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

print(f"SPLIT_ID   : {SPLIT_ID} / {N_SPLITS}")
print(f"Output dir : {OUTPUT_DIR}")
print(f"Report dir : {REPORT_DIR}")


SPLIT_ID   : 4 / 15
Output dir : /content/drive/MyDrive/kindle_label/ATE_Phase_1/split_04
Report dir : /content/drive/MyDrive/kindle_label/ATE_Phase_1/split_04/reports/ate_1


In [8]:
import sys
if SCRIPT_DIR not in sys.path:
    sys.path.insert(0, SCRIPT_DIR)


from extract_aspect_phase1 import predict_aspects

print("extract_aspect_phase1 loaded OK")

Loading weights:   0%|          | 0/201 [00:01<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:01<?, ?it/s]

extract_aspect_phase1 loaded OK


In [9]:
import pandas as pd
import json

# Load split assignment từ JSON (tạo bởi prepare_splits.ipynb)
assignment_path = f"{SPLITS_DIR}/split_assignment_{N_SPLITS}.json"
assert os.path.exists(assignment_path), (
    f"Không tìm thấy {assignment_path}\n"
    f"→ Hãy chạy prepare_splits.ipynb trước!"
)

with open(assignment_path, "r", encoding="utf-8") as f:
    all_assignments = json.load(f)

split_entries = all_assignments[str(SPLIT_ID)]
assert split_entries, f"Split {SPLIT_ID} rỗng!"

print(f"Split {SPLIT_ID}/{N_SPLITS} — {len(split_entries)} category(s):")
for entry in split_entries:
    print(f"  {entry['category']}: {len(entry['files']):,} part file(s)")


Split 4/15 — 1 category(s):
  cleaned2_Kindle_Store: 27 part file(s)


In [10]:
import torch


def split_confidence(sub: pd.DataFrame, results: pd.DataFrame):
    """
    Ghép kết quả ATE vào sub, phân loại high/low confidence.

    High confidence:
      - gate_confidence <= NEGATIVE_UPPER  (negative signal)
      - gate_confidence >= POSITIVE_LOWER AND len(aspects) >= 1  (positive signal)
    Low confidence: còn lại
    """
    sub = sub.reset_index(drop=True).copy()
    sub["gate_confidence"] = results["gate_confidence"].values
    sub["aspects"]         = results["aspects"].values
    sub["confidences"]     = results["confidences"].values

    is_negative = sub["gate_confidence"] <= NEGATIVE_UPPER
    is_positive = (
        (sub["gate_confidence"] >= POSITIVE_LOWER) &
        (sub["aspects"].apply(len) >= 1)
    )
    is_high = is_negative | is_positive

    return sub[is_high].copy(), sub[~is_high].copy()


def write_report(category: str, n_total: int, high: pd.DataFrame, low: pd.DataFrame):
    """Ghi thống kê high confidence ra file txt."""
    n_high = len(high)
    n_low  = len(low)

    has_aspect_mask = high["aspects"].apply(len) >= 1
    n_positive      = has_aspect_mask.sum()
    n_negative      = n_high - n_positive

    all_aspects    = high["aspects"].explode().dropna()
    total_aspects  = len(all_aspects[all_aspects != ""])  if len(all_aspects) > 0 else 0
    avg_per_sample = total_aspects / n_high if n_high > 0 else 0.0
    avg_per_pos    = total_aspects / n_positive if n_positive > 0 else 0.0
    has_aspect_rate = n_positive / n_high * 100 if n_high > 0 else 0.0

    lines = [
        f"=" * 60,
        f"ATE Phase 1 Report — {category}",
        f"=" * 60,
        f"",
        f"[Dataset]",
        f"  Total samples processed : {n_total:>8,}",
        f"  High confidence         : {n_high:>8,}  ({n_high/n_total*100:.2f}% of total)",
        f"    - Negative (no aspect): {n_negative:>8,}  ({n_negative/n_high*100:.2f}% of high conf)",
        f"    - Positive (has aspect): {n_positive:>7,}  ({n_positive/n_high*100:.2f}% of high conf)",
        f"  Low confidence          : {n_low:>8,}  ({n_low/n_total*100:.2f}% of total)",
        f"",
        f"[Thresholds]",
        f"  gate_threshold  : {GATE_THRESHOLD}",
        f"  span_threshold  : {SPAN_THRESHOLD}",
        f"  negative_upper  : <= {NEGATIVE_UPPER}",
        f"  positive_lower  : >= {POSITIVE_LOWER}",
        f"",
        f"[High Confidence Statistics]",
        f"  Has aspect rate         : {has_aspect_rate:.2f}%",
        f"  Total aspects extracted : {total_aspects:>8,}",
        f"  Avg aspects / sample    : {avg_per_sample:.4f}",
        f"  Avg aspects / positive  : {avg_per_pos:.4f}",
        f"",
        f"[Output files]",
        f"  high: {OUTPUT_DIR}/high_confidence_samples_{category}.parquet",
        f"  low : {OUTPUT_DIR}/low_confidence_samples_{category}.parquet",
        f"=" * 60,
    ]

    report_path = f"{REPORT_DIR}/ate_1_report_{category}.txt"
    with open(report_path, "w", encoding="utf-8") as f:
        f.write("\n".join(lines))

    print(f"  Report saved : {report_path}")
    # In tóm tắt ra console
    print(f"  high={n_high:,}  low={n_low:,}  has_aspect={has_aspect_rate:.1f}%  "
          f"total_aspects={total_aspects:,}  avg/sample={avg_per_sample:.3f}")


print("Helper functions defined.")

Helper functions defined.


In [11]:
from tqdm.auto import tqdm
import gc
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as pad
import queue, threading


device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device        : {device}")
if device == "cpu":
    print(" WARNING: GPU không được dùng ")
    _GATE_BS = 128
    _ATE_BS  = 64
else:
    print(f"GPU           : {torch.cuda.get_device_name(0)}")
    print(f"GPU VRAM      : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
    _GATE_BS = 512
    _ATE_BS  = 256

CHUNK_SIZE = 50_000
print(f"CHUNK_SIZE      : {CHUNK_SIZE:,}")
print(f"Gate batch size : {_GATE_BS}")
print(f"ATE  batch size : {_ATE_BS}")

HIGH_COLS = [
    "parent_asin", "sentence_id", "sentence_text",
    "rating", "category_name",
    "gate_confidence", "aspects", "confidences",
]
LOW_COLS = [
    "parent_asin", "sentence_id", "sentence_text",
    "rating", "gate_confidence",
]

def filter_cols(df, cols):
    """Chỉ lấy các cột thực sự tồn tại trong df."""
    return df[[c for c in cols if c in df.columns]]

def iter_chunks_prefetch(ds, chunk_size, prefetch=2):
    q = queue.Queue(maxsize=prefetch)
    def _producer():
        try:
            for batch in ds.scanner(batch_size=chunk_size).to_batches():
                q.put(batch.to_pandas())
                del batch
        finally:
            q.put(None)
    threading.Thread(target=_producer, daemon=True).start()
    while True:
        item = q.get()
        if item is None:
            break
        yield item

# ── Vòng lặp chính — dùng split_entries ─────────────────
for entry in split_entries:
    category   = entry["category"]
    part_files = entry["files"]

    # Tạo dataset từ danh sách part files của split này
    ds      = pad.dataset(part_files, format="parquet")
    n_total = ds.count_rows()

    print(f"\n{'─'*55}")
    print(f" Processing : {category}  ({n_total:,} samples)")
    print(f"  Part files : {len(part_files):,}")
    print(f"  Split      : {SPLIT_ID}/{N_SPLITS}")

    out_high = f"{OUTPUT_DIR}/high_confidence_samples_{category}.parquet"
    out_low  = f"{OUTPUT_DIR}/low_confidence_samples_{category}.parquet"

    if os.path.exists(out_high) and os.path.exists(out_low):
        print(f"  [SKIP] Already done.")
        continue

    writer_high = writer_low = None
    total_high  = total_low  = 0
    n_chunks    = (n_total + CHUNK_SIZE - 1) // CHUNK_SIZE

    try:
        for chunk in tqdm(
            iter_chunks_prefetch(ds, CHUNK_SIZE),
            total=n_chunks,
            desc=f"  split{SPLIT_ID:02d}/{category}",
        ):
            results = predict_aspects(
                texts=chunk["sentence_text"].tolist(),
                gate_threshold=GATE_THRESHOLD,
                span_threshold=SPAN_THRESHOLD,
                gate_batch_size=_GATE_BS,
                ate_batch_size=_ATE_BS,
                max_length=MAX_LENGTH,
            )

            high_df, low_df = split_confidence(chunk, results)
            del chunk, results

            high_table = pa.Table.from_pandas(filter_cols(high_df, HIGH_COLS), preserve_index=False)
            low_table  = pa.Table.from_pandas(filter_cols(low_df,  LOW_COLS),  preserve_index=False)

            if writer_high is None:
                writer_high = pq.ParquetWriter(out_high, high_table.schema)
                writer_low  = pq.ParquetWriter(out_low,  low_table.schema)

            writer_high.write_table(high_table)
            writer_low.write_table(low_table)

            total_high += len(high_df)
            total_low  += len(low_df)

            del high_df, low_df, high_table, low_table
            gc.collect()

    finally:
        if writer_high: writer_high.close()
        if writer_low:  writer_low.close()

    print(f"  Saved high : {out_high}  ({total_high:,} rows)")
    print(f"  Saved low  : {out_low}  ({total_low:,} rows)")

    high_df_final = pd.read_parquet(out_high)
    low_df_final  = pd.read_parquet(out_low)
    write_report(category, n_total, high_df_final, low_df_final)
    del high_df_final, low_df_final
    gc.collect()

print(f"\n{'═'*55}")
print(f" Split {SPLIT_ID} done.")

Device        : cuda
GPU           : Tesla T4
GPU VRAM      : 15.6 GB
CHUNK_SIZE      : 50,000
Gate batch size : 512
ATE  batch size : 256

───────────────────────────────────────────────────────
 Processing : cleaned2_Kindle_Store  (4,111,005 samples)
  Part files : 27
  Split      : 4/15


  split04/cleaned2_Kindle_Store:   0%|          | 0/83 [00:00<?, ?it/s]

  Saved high : /content/drive/MyDrive/kindle_label/ATE_Phase_1/split_04/high_confidence_samples_cleaned2_Kindle_Store.parquet  (3,238,077 rows)
  Saved low  : /content/drive/MyDrive/kindle_label/ATE_Phase_1/split_04/low_confidence_samples_cleaned2_Kindle_Store.parquet  (872,928 rows)
  Report saved : /content/drive/MyDrive/kindle_label/ATE_Phase_1/split_04/reports/ate_1/ate_1_report_cleaned2_Kindle_Store.txt
  high=3,238,077  low=872,928  has_aspect=50.1%  total_aspects=2,101,829  avg/sample=0.649

═══════════════════════════════════════════════════════
 Split 4 done.


In [12]:
summary_rows = []
for entry in split_entries:
    category = entry["category"]

    high_path = f"{OUTPUT_DIR}/high_confidence_samples_{category}.parquet"
    low_path  = f"{OUTPUT_DIR}/low_confidence_samples_{category}.parquet"

    if not os.path.exists(high_path):
        print(f"[MISSING] {high_path}")
        continue

    high = pd.read_parquet(high_path)
    low  = pd.read_parquet(low_path)

    n_total    = len(high) + len(low)
    n_positive = (high["aspects"].apply(len) >= 1).sum()
    n_negative = len(high) - n_positive

    all_asp   = high["aspects"].explode().dropna()
    total_asp = len(all_asp[all_asp != ""]) if len(all_asp) > 0 else 0

    summary_rows.append({
        "category"       : category,
        "total"          : n_total,
        "high_conf"      : len(high),
        "high_positive"  : int(n_positive),
        "high_negative"  : int(n_negative),
        "low_conf"       : len(low),
        "high_rate_%"    : round(len(high) / n_total * 100, 2),
        "has_asp_rate_%" : round(n_positive / len(high) * 100, 2) if len(high) > 0 else 0,
        "total_aspects"  : total_asp,
        "avg_asp/sample" : round(total_asp / len(high), 4) if len(high) > 0 else 0,
    })

summary_df = pd.DataFrame(summary_rows)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
print(summary_df.to_string(index=False))


             category   total  high_conf  high_positive  high_negative  low_conf  high_rate_%  has_asp_rate_%  total_aspects  avg_asp/sample
cleaned2_Kindle_Store 4111005    3238077        1622046        1616031    872928        78.77           50.09        2101829          0.6491


In [13]:
# Kiểm tra file output của category đầu tiên
CHECK_CATEGORY = split_entries[0]["category"]
high_sample = pd.read_parquet(f"{OUTPUT_DIR}/high_confidence_samples_{CHECK_CATEGORY}.parquet")
low_sample  = pd.read_parquet(f"{OUTPUT_DIR}/low_confidence_samples_{CHECK_CATEGORY}.parquet")

print(f"=== {CHECK_CATEGORY} ===")
print(f"High shape : {high_sample.shape}")
print(f"Low  shape : {low_sample.shape}")
print(f"\nHigh columns : {list(high_sample.columns)}")
print(f"Low  columns : {list(low_sample.columns)}")

print("\n--- High confidence sample (positive) ---")
pos = high_sample[high_sample["aspects"].apply(len) >= 1]
print(pos[["sentence_text", "gate_confidence", "aspects", "confidences"]].head(3).to_string())

print("\n--- High confidence sample (negative) ---")
neg = high_sample[high_sample["aspects"].apply(len) == 0]
print(neg[["sentence_text", "gate_confidence", "aspects"]].head(3).to_string())

print("\n--- Low confidence sample ---")
print(low_sample[["sentence_text", "gate_confidence"]].head(3).to_string())


=== cleaned2_Kindle_Store ===
High shape : (3238077, 7)
Low  shape : (872928, 5)

High columns : ['parent_asin', 'sentence_id', 'sentence_text', 'rating', 'gate_confidence', 'aspects', 'confidences']
Low  columns : ['parent_asin', 'sentence_id', 'sentence_text', 'rating', 'gate_confidence']

--- High confidence sample (positive) ---
                                                                                           sentence_text  gate_confidence    aspects           confidences
8                                                            these stories are such a quick and fun read         0.991170  [stories]  [0.9844422936439514]
9                                                               this series is still alot of fun to read         0.987448   [series]  [0.9819855690002441]
10  i know alot people give these books bad [GENERIC_NOUN] some of the books are so silly they are funny         0.993268    [books]  [0.9617369174957275]

--- High confidence sample (negative) ---
  